# Surrogate Model Convergence
This notebook provides a template for checking and plotting the convergence of surrogate models based on the number of points sampled randomly.
Multiple random samples are taken from each function to check method accuracy relative to number of samples.

In [2]:
%load_ext autoreload
%autoreload 2
#%matplotlib widget

import numpy as np
import matplotlib.pyplot as plt
import multi_fidelity_functions as mfs
from SurrogateModels import kriging

### Convergence Over Different Dimensions
Test a set of functions from 1-8 dimensions for accuracy in fitting a function

In [4]:
# Set some storage ontainers
error = {}
points = {}

In [6]:
def create_test_points(func: mfs.MultiFidelityTemplate, n=2000):
    # Create a set of samples to create a ground truth
    X_test = func.sample_random(n)
    y_test = func.high_fidelity(X_test)
    return X_test, y_test


def generate_model_comparisons(
    name,
    func: mfs.MultiFidelityTemplate,
    X_test,
    y_test,
    sample_low=5,
    sample_high=105,
    step_size=5,
    seeds=5,
    model_error = {},
    points = {}
):
    # Generate a Kriging model of the data and test accuracy using a number of random sample points
    n_points = list(range(sample_low, sample_high, step_size))
    points[name] = []
    model_error[name] = []
    for seed in range(0, seeds):
        for n in n_points:
            X_train = func.sample_random(n, seed=seed)
            y_train = func.high_fidelity(X_train)            
            model = kriging.Kriging(X_train, y_train, nugget=1e-7)
            model_error[name].append(model.model_error(X_test, y_test))
            points[name].append(n)
            print(f"Run {name}: seed {seed}/points {n} complete")

def plot_error_measure(points, error, name, color="blue", marker = "o", label = "Test",ax=None):
    if ax == None:
        fig = plt.figure(name)
        ax = fig.add_subplot()
    ax.set_title(name)
    ax.scatter(
        points,
        error,
        color=color,
        label=label,
        marker=marker,
    )
    ax.set_xlabel("Samples")
    ax.set_ylabel("RMS Error")
    ax.legend()
    ax.grid()
    ax.set_yscale("log")
    return ax

In [ ]:
# key = "1D"
# my_function = mfs.increasing_dimensions[key]()
# X_test, y_test = create_test_points(my_function, n=2000)
# generate_model_comparisons(
#     key,
#     my_function,
#     X_test=X_test,
#     y_test=y_test,
#     sample_low=5,
#     sample_high=150,
#     step_size=5,
#     seeds=5,
#     model_error=error,
#     points=points,
# )

In [ ]:
# # Output the results to a csv
# import csv
# output_file = f"Convergence\KrigingResults_{key}_small.csv"
# with open(output_file,"w", newline='') as file:
#     writer = csv.writer(file)
#     for i, val in enumerate(points[key]):
#         writer.writerow([val,points[key][i],model_error[key][i],training_time[key][i],evaluation_time[key][i]])

In [ ]:
# ax = plot_error_measure(points[key],error[key],name = key)


In [4]:
# Try running cokriging with error comparisons to the above
import cokriging
import time
import itertools


def generate_cokriging_model_comparisons(
    name,
    func: mfs.MultiFidelityTemplate,
    X_test,
    y_test,
    sample_lf=(5, 60),
    sample_hf=(5, 120),
    step_lf=5,
    step_hf=5,
    seeds=5,
    model_error={},
    points_lf={},
    points_hf={},
    training_time={},
    evaluation_time={},
):
    # Generate a Kriging model of the data and test accuracy using a number of random sample points
    n_lf_range = list(range(*sample_lf, step_lf))
    n_hf_range = list(range(*sample_hf, step_hf))
    points_lf[name] = []
    points_hf[name] = []
    training_time[name] = []
    evaluation_time[name] = []
    model_error[name] = []

    for nh, nl, seed in itertools.product(n_hf_range, n_lf_range, range(seeds)):

        # define training points by randomly sampling functions
        X_train_lf = func.sample_random(nl, seed=seed)
        y_train_lf = func.low_fidelity(X_train_lf)
        X_train_hf = func.sample_random(nh, seed=seed)
        y_train_hf = func.high_fidelity(X_train_hf)

        # Setup timer to gauge model training time
        t0 = time.perf_counter()

        # Train the model
        if nl == 0:
            model = kriging.Kriging(x_mat=X_train_hf, y_vec=y_train_hf, nugget=1e-7,scale_input=True,bounds=func.bounds)
        else:
            model = cokriging.CoKriging(
                x_lf=X_train_lf,
                y_lf=y_train_lf,
                x_hf=X_train_hf,
                y_hf=y_train_hf,
                nugget=1e-6,
                scale_input=True,
                bounds=func.bounds
            )
        model_training_time = time.perf_counter() - t0

        # Calculate error using a large number of model samples
        t0 = time.perf_counter()
        error = model.model_error(X_test, y_test)
        model_evaluation_time = time.perf_counter() - t0

        model_error[name].append(error)
        points_lf[name].append(nl)
        points_hf[name].append(nh)
        training_time[name].append(model_training_time)
        evaluation_time[name].append(model_evaluation_time)

        print(
            f"Run {name} :: Seed {seed} :: n_hf {nh} :: n_lf {nl} :: Error {error} :: t_train {model_training_time}"
        )


def plot_cokriging_data(
    points_lf,
    points_hf,
    output,
    name="Cokriging Data",
    color="black",
    label="data",
    log_output=False,
    ax=None,
):
    if ax == None:
        fig = plt.figure(name)
        ax = fig.add_subplot(projection="3d")
    ax.set_title(name)
    ax.set_xlabel("Samples LF")
    ax.set_ylabel("Samples HF")
    ax.set_zlabel(label)

    if log_output:
        output = np.log10(output)

    ax.scatter(
        points_lf,
        points_hf,
        output,
        color=color,
        label=label,
    )


def plot_cokriging_data_slice(
    points_lf,
    points_hf,
    output,
    fix_lf=True,
    fix_value=10,
    name="Cokriging Data",
    color="black",
    label="data",
    log_output=False,
    ax=None,
):
    if ax == None:
        fig = plt.figure(name)
        ax = fig.add_subplot()
    ax.set_title(name)
    x_values = []
    y_values = []
    if fix_lf:
        ax.set_xlabel("Samples HF")
        for i, val in enumerate(points_lf):
            if val == fix_value:
                x_values.append(points_hf[i])
                y_values.append(output[i])
    else:
        ax.set_xlabel("Samples_LF")
        for i, val in enumerate(points_hf):
            if val == fix_value:
                x_values.append(points_lf[i])
                y_values.append(output[i])
    ax.set_ylabel(label)

    if log_output:
        y_values = np.log10(y_values)

    print(x_values)
    ax.scatter(
        x_values,
        y_values,
        color=color,
        label=label,
    )
    return ax

In [8]:
# Set storage containers
model_error = {}
points_lf = {}
points_hf = {}
training_time = {}
evaluation_time = {}

In [41]:
key = "1D"
my_function = mfs.increasing_dimensions[key]()
X_test, y_test = create_test_points(my_function, n=2000)
generate_cokriging_model_comparisons(
    key,
    my_function,
    X_test=X_test,
    y_test=y_test,
    sample_lf=(0, 30),
    sample_hf=(4, 30),
    step_lf=2,
    step_hf=2,
    seeds=3,
    model_error = model_error,
    points_lf = points_lf,
    points_hf = points_hf,
    training_time = training_time,
    evaluation_time = evaluation_time,
)

Run 1D :: Seed 0 :: n_hf 4 :: n_lf 0 :: Error 5.022183244742917 :: t_train 0.004960199818015099
Run 1D :: Seed 1 :: n_hf 4 :: n_lf 0 :: Error 4.455996607556095 :: t_train 0.004888599971309304
Run 1D :: Seed 2 :: n_hf 4 :: n_lf 0 :: Error 5.158202097644533 :: t_train 0.00401380006223917
Run 1D :: Seed 0 :: n_hf 4 :: n_lf 2 :: Error 4.058596047277122 :: t_train 0.02809839998371899
Run 1D :: Seed 1 :: n_hf 4 :: n_lf 2 :: Error 4.513628374814454 :: t_train 0.03892030008137226
Run 1D :: Seed 2 :: n_hf 4 :: n_lf 2 :: Error 5.7166659699964555 :: t_train 0.03562420001253486
Run 1D :: Seed 0 :: n_hf 4 :: n_lf 4 :: Error 8.76701832683161 :: t_train 0.07141599990427494
Run 1D :: Seed 1 :: n_hf 4 :: n_lf 4 :: Error 4.319813455968139 :: t_train 0.11788390018045902
Run 1D :: Seed 2 :: n_hf 4 :: n_lf 4 :: Error 8.368113044827203 :: t_train 0.078502899967134
Run 1D :: Seed 0 :: n_hf 4 :: n_lf 6 :: Error 1.5757065151888707 :: t_train 0.1042654998600483
Run 1D :: Seed 1 :: n_hf 4 :: n_lf 6 :: Error 3.72

In [9]:
import csv

def export_to_csv(key,output_file,points_lf,points_hf,model_error,training_time,evaluation_time):
    with open(output_file,"w", newline='') as file:
        writer = csv.writer(file)
        for i, val in enumerate(points_lf[key]):
            writer.writerow([val,points_hf[key][i],model_error[key][i],training_time[key][i],evaluation_time[key][i]])

In [43]:
# # Output the results to a csv
# import csv
# output_file = f"CokrigingResults_{key}_small.csv"
# with open(output_file,"w", newline='') as file:
#     writer = csv.writer(file)
#     for i, val in enumerate(points_lf[key]):
#         writer.writerow([val,points_hf[key][i],model_error[key][i],training_time[key][i],evaluation_time[key][i]])

output_file = f"CokrigingResults_{key}_042926.csv"
export_to_csv(key,output_file,points_lf,points_hf,model_error,training_time,evaluation_time)

In [7]:
key = "2D"
my_function = mfs.increasing_dimensions[key]()
X_test, y_test = create_test_points(my_function, n=2000)
generate_cokriging_model_comparisons(
    key,
    my_function,
    X_test=X_test,
    y_test=y_test,
    sample_lf=(0, 400),
    sample_hf=(20, 200),
    step_lf=10,
    step_hf=10,
    seeds=3,
    model_error = model_error,
    points_lf = points_lf,
    points_hf = points_hf,
    training_time = training_time,
    evaluation_time = evaluation_time,
)

Run 2D :: Seed 0 :: n_hf 20 :: n_lf 0 :: Error 1.3899697094434451 :: t_train 0.8723130999132991
Run 2D :: Seed 1 :: n_hf 20 :: n_lf 0 :: Error 1.4373972294972177 :: t_train 1.2807725998573005
Run 2D :: Seed 2 :: n_hf 20 :: n_lf 0 :: Error 1.3966571336668594 :: t_train 1.408702600048855
Run 2D :: Seed 0 :: n_hf 20 :: n_lf 10 :: Error 1.7565749332243947 :: t_train 0.9357955001760274
Run 2D :: Seed 1 :: n_hf 20 :: n_lf 10 :: Error 1.889914808040818 :: t_train 1.1329846000298858
Run 2D :: Seed 2 :: n_hf 20 :: n_lf 10 :: Error 1.8061878393448214 :: t_train 0.8055090000852942


KeyboardInterrupt: 

In [45]:
output_file = f"CokrigingResults_{key}__042926.csv"
export_to_csv(key,output_file,points_lf,points_hf,model_error,training_time,evaluation_time)

In [5]:
key = "4D"
my_function = mfs.increasing_dimensions[key]()
X_test, y_test = create_test_points(my_function, n=2000)
generate_cokriging_model_comparisons(
    key,
    my_function,
    X_test=X_test,
    y_test=y_test,
    sample_lf=(0, 1000),
    sample_hf=(40, 500),
    step_lf=100,
    step_hf=40,
    seeds=1,
    model_error = model_error,
    points_lf = points_lf,
    points_hf = points_hf,
    training_time = training_time,
    evaluation_time = evaluation_time,
)

NameError: name 'create_test_points' is not defined

In [ ]:
output_file = f"CokrigingResults_{key}__050426.csv"
export_to_csv(key,output_file,points_lf,points_hf,model_error,training_time,evaluation_time)

In [7]:
key = "6D"
my_function = mfs.increasing_dimensions[key]()
X_test, y_test = create_test_points(my_function, n=2000)
print(y_test)
# generate_cokriging_model_comparisons(
#     key,
#     my_function,
#     X_test=X_test,
#     y_test=y_test,
#     sample_lf=(0, 1000),
#     sample_hf=(40, 500),
#     step_lf=100,
#     step_hf=40,
#     seeds=3,
#     model_error = model_error,
#     points_lf = points_lf,
#     points_hf = points_hf,
#     training_time = training_time,
#     evaluation_time = evaluation_time,
# )

[[1.92387221e+04]
 [1.23742562e+05]
 [4.32240774e+01]
 ...
 [1.09689945e+06]
 [5.90557589e+04]
 [4.70216268e+02]]


In [ ]:
import csv

def export_data(X_test, y_test, filename):
    with open(filename,"w", newline='') as file:
        writer = csv.writer(file)
        for i, val in enumerate(y_test):
            writer.writerow([X_test[i],val,filename])

In [ ]:
export_data(X_test,y_test,filename)

In [25]:
output_file = f"CokrigingResults_{key}__050426.csv"
export_to_csv(key,output_file,points_lf,points_hf,model_error,training_time,evaluation_time)

In [11]:
key = "8D"
my_function = mfs.increasing_dimensions[key]()
X_test, y_test = create_test_points(my_function, n=2000)
generate_cokriging_model_comparisons(
    key,
    my_function,
    X_test=X_test,
    y_test=y_test,
    sample_lf=(0, 1000),
    sample_hf=(40, 500),
    step_lf=100,
    step_hf=40,
    seeds=3,
    model_error = model_error,
    points_lf = points_lf,
    points_hf = points_hf,
    training_time = training_time,
    evaluation_time = evaluation_time,
)

Run 8D :: Seed 0 :: n_hf 40 :: n_lf 0 :: Error 1.1534586013330843 :: t_train 73.24740169988945
Run 8D :: Seed 1 :: n_hf 40 :: n_lf 0 :: Error 0.9528483157256499 :: t_train 51.68677089991979
Run 8D :: Seed 2 :: n_hf 40 :: n_lf 0 :: Error 1.121689378768496 :: t_train 74.12920110020787
Run 8D :: Seed 0 :: n_hf 40 :: n_lf 100 :: Error 0.48614731070476275 :: t_train 342.9417745000683
Run 8D :: Seed 1 :: n_hf 40 :: n_lf 100 :: Error 0.572150458520937 :: t_train 454.6425233001355
Run 8D :: Seed 2 :: n_hf 40 :: n_lf 100 :: Error 0.4518412607164003 :: t_train 536.3014020998962
Run 8D :: Seed 0 :: n_hf 40 :: n_lf 200 :: Error 0.14319481488831062 :: t_train 893.7709441999905
Run 8D :: Seed 1 :: n_hf 40 :: n_lf 200 :: Error 0.2651972605374257 :: t_train 1011.0731681000907
Run 8D :: Seed 2 :: n_hf 40 :: n_lf 200 :: Error 0.19426648717775463 :: t_train 1082.6553946998902
Run 8D :: Seed 0 :: n_hf 40 :: n_lf 300 :: Error 0.1361812248056516 :: t_train 1970.5586260999553
Run 8D :: Seed 1 :: n_hf 40 :: n

KeyboardInterrupt: 

In [ ]:
output_file = f"CokrigingResults_{key}__050426.csv"
export_to_csv(key,output_file,points_lf,points_hf,model_error,training_time,evaluation_time)